# Modelo barotrópico con difusión de potencia: solución temporal numérica

Se estudia el modelo

\[
p=\omega\rho,
\qquad
Q=\alpha\rho^\beta,
\]

con las ecuaciones

\[
3H^2=\rho+\alpha\rho^\beta,
\]

\[
\dot H=-\frac{1+\omega}{2}\rho,
\]

\[
\dot\rho=
-\frac{3H(1+\omega)\rho}
{1+\alpha\beta\rho^{\beta-1}},
\qquad
\dot a=aH.
\]

El método `solve_background_t` integra las variables logarítmicas

\[
x=\ln\rho,
\qquad
N=\ln a,
\]

y devuelve únicamente

\[
t,\quad \rho(t),\quad a(t),\quad H(t),\quad \dot H(t).
\]

## Estado del trabajo

- [x] Implementación del solucionador temporal en `src/models.py`.
- [x] Validación contra la solución exacta para \(\beta=1\).
- [x] Validación interna para \(\beta\neq1\).
- [ ] Validación sistemática de los eventos de detención.
- [ ] Estudio matemático de los intervalos de parámetros.
- [ ] Diseño e implementación del clasificador.
- [ ] Gráficos finales por intervalos físicos.


## 1. Configuración del proyecto

La siguiente celda funciona tanto si el notebook se ejecuta desde la raíz del proyecto como desde la carpeta `notebooks`.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

current_path = Path.cwd().resolve()

if (current_path / "src").exists():
    PROJECT_ROOT = current_path
elif (current_path.parent / "src").exists():
    PROJECT_ROOT = current_path.parent
else:
    raise FileNotFoundError(
        "No se encontró la carpeta src. "
        "Ejecuta el notebook dentro del proyecto cosmo_singularities."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import (
    BarotropicPowerDiffusionParams,
    BarotropicPowerDiffusionModel,
)

if not hasattr(BarotropicPowerDiffusionModel, "solve_background_t"):
    raise AttributeError(
        "BarotropicPowerDiffusionModel no contiene solve_background_t. "
        "Agrega primero el método numérico en src/models.py."
    )

print("Project root:", PROJECT_ROOT)


Project root: /Users/macbook/Usach/TesisxD/cosmo_singularities


## 2. Validación analítica para \(\beta=1\)

Este caso tiene solución exacta y permite validar simultáneamente \(\rho(t)\), \(a(t)\), \(H(t)\) y \(\dot H(t)\).

In [3]:
params_test = BarotropicPowerDiffusionParams(
    omega=-1.2,
    alpha=0.3,
    beta=1.0,
    rho0=1.0,
    a0=1.0,
    t0=0.0,
)

model_test = BarotropicPowerDiffusionModel(params_test)

In [4]:
background_num = model_test.solve_background_t(
    t_end=2.0,
    n_points=2000,
    rho_max=1e12,
    a_max=1e12,
)

In [5]:
print(type(model_test))
print(type(background_num))

<class 'src.models.BarotropicPowerDiffusionModel'>
<class 'pandas.core.frame.DataFrame'>


In [6]:
model_params = model_test.params
initial_state = background_num.iloc[0]

rho0_expected = model_params.rho0
a0_expected = model_params.a0

H0_expected = np.sqrt(
    (
        model_params.rho0
        + model_params.alpha
        * model_params.rho0**model_params.beta
    )
    / 3.0
)

Hdot0_expected = (
    -0.5
    * (1.0 + model_params.omega)
    * model_params.rho0
)

initial_checks = {
    "rho(t0)": np.isclose(
        initial_state["rho"],
        rho0_expected,
        rtol=1e-10,
        atol=1e-12,
    ),
    "a(t0)": np.isclose(
        initial_state["a"],
        a0_expected,
        rtol=1e-10,
        atol=1e-12,
    ),
    "H(t0)": np.isclose(
        initial_state["H"],
        H0_expected,
        rtol=1e-10,
        atol=1e-12,
    ),
    "Hdot(t0)": np.isclose(
        initial_state["Hdot"],
        Hdot0_expected,
        rtol=1e-10,
        atol=1e-12,
    ),
}

initial_checks

{'rho(t0)': np.True_,
 'a(t0)': np.True_,
 'H(t0)': np.True_,
 'Hdot(t0)': np.True_}

In [7]:
# Parámetros del modelo utilizado en la prueba
params = model_test.params

if not np.isclose(params.beta, 1.0):
    raise ValueError(
        "Esta validación analítica requiere beta = 1."
    )

t_num = background_num["t"].to_numpy()

rho0 = params.rho0
a0 = params.a0
t0 = params.t0
omega = params.omega
alpha = params.alpha

# Valor inicial del parámetro de Hubble
H0 = np.sqrt(
    (1.0 + alpha) * rho0 / 3.0
)

# Factor común de la solución analítica
base = (
    1.0
    + (
        3.0
        * (1.0 + omega)
        * H0
        / (2.0 * (1.0 + alpha))
    )
    * (t_num - t0)
)

if np.any(base <= 0):
    raise ValueError(
        "La grilla temporal alcanzó o sobrepasó "
        "la singularidad de la solución analítica."
    )

# Soluciones analíticas
rho_analytic = rho0 * base**(-2.0)

a_analytic = a0 * base ** (
    2.0
    * (1.0 + alpha)
    / (3.0 * (1.0 + omega))
)

H_analytic = H0 / base

Hdot_analytic = (
    -0.5
    * (1.0 + omega)
    * rho_analytic
)

In [8]:
def maximum_relative_error(
    numerical_values,
    analytical_values,
):
    """
    Calcula el máximo error relativo entre dos arreglos.
    """

    numerical_values = np.asarray(
        numerical_values,
        dtype=float,
    )

    analytical_values = np.asarray(
        analytical_values,
        dtype=float,
    )

    denominator = np.maximum(
        np.abs(analytical_values),
        np.finfo(float).tiny,
    )

    relative_error = (
        np.abs(
            numerical_values
            - analytical_values
        )
        / denominator
    )

    return float(np.max(relative_error))

In [9]:
validation_errors = {
    "rho(t)": maximum_relative_error(
        background_num["rho"].to_numpy(),
        rho_analytic,
    ),
    "a(t)": maximum_relative_error(
        background_num["a"].to_numpy(),
        a_analytic,
    ),
    "H(t)": maximum_relative_error(
        background_num["H"].to_numpy(),
        H_analytic,
    ),
    "Hdot(t)": maximum_relative_error(
        background_num["Hdot"].to_numpy(),
        Hdot_analytic,
    ),
}

validation_errors

{'rho(t)': 7.332887385794746e-10,
 'a(t)': 1.588791002664966e-09,
 'H(t)': 3.6664438927229267e-10,
 'Hdot(t)': 7.332887092867276e-10}

In [10]:
validation_table = pd.DataFrame(
    {
        "cantidad": list(
            validation_errors.keys()
        ),
        "error_relativo_maximo": list(
            validation_errors.values()
        ),
    }
)

validation_table

,cantidad,error_relativo_maximo
0,rho(t),7.332887e-10
1,a(t),1.588791e-09
2,H(t),3.666444e-10
3,Hdot(t),7.332887e-10


In [11]:
validation_tolerance = 1e-7

validation_checks = {
    quantity: error < validation_tolerance
    for quantity, error in validation_errors.items()
}

validation_checks

{'rho(t)': True, 'a(t)': True, 'H(t)': True, 'Hdot(t)': True}

In [12]:
solver_validated = all(
    validation_checks.values()
)

print(
    "Solucionador validado:",
    solver_validated,
)

Solucionador validado: True


## 3. Validación general para \(\beta\neq1\)

Como no se dispone de una inversión temporal cerrada general, se verifican la relación integrada ρ(a), Friedmann, Raychaudhuri y las dos ecuaciones diferenciales.

In [13]:
params_general = BarotropicPowerDiffusionParams(
    omega=-1.2,
    alpha=0.3,
    beta=1.5,
    rho0=1.0,
    a0=1.0,
    t0=0.0,
)

model_general = BarotropicPowerDiffusionModel(
    params_general
)

background_general = model_general.solve_background_t(
    t_end=1.0,
    n_points=4000,
    rho_max=1e12,
    a_max=1e12,
    max_step=0.005,
)

In [14]:
print(background_general.attrs)

background_general.tail()

{'solver_success': True, 'solver_message': 'The solver successfully reached the end of the integration interval.', 'termination_event': None, 'termination_time': 1.0, 'requested_t_end': 1.0, 'reached_t_end': True, 'method': 'DOP853', 'rtol': 1e-09, 'atol': 1e-11, 'rho_min': 1e-12, 'rho_max': 1000000000000.0, 'a_max': 1000000000000.0, 'boundary_tol': 1e-08}


,t,rho,a,H,Hdot
3995,0.99900,1.333863,2.039137,0.773739,0.133386
3996,0.99925,1.333965,2.039532,0.773773,0.133396
3997,0.99950,1.334067,2.039926,0.773806,0.133407
3998,0.99975,1.334169,2.040321,0.773840,0.133417
3999,1.00000,1.334271,2.040716,0.773873,0.133427


In [15]:
initial_state_general = background_general.iloc[0]

general_initial_checks = {
    "rho(t0)": np.isclose(
        initial_state_general["rho"],
        params_general.rho0,
    ),
    "a(t0)": np.isclose(
        initial_state_general["a"],
        params_general.a0,
    ),
    "H(t0)": np.isclose(
        initial_state_general["H"],
        model_general.hubble(
            params_general.rho0
        ),
    ),
    "Hdot(t0)": np.isclose(
        initial_state_general["Hdot"],
        model_general.hdot(
            params_general.rho0
        ),
    ),
}

general_initial_checks

{'rho(t0)': np.True_,
 'a(t0)': np.True_,
 'H(t0)': np.True_,
 'Hdot(t0)': np.True_}

In [16]:
rho_general = background_general["rho"].to_numpy()
a_general = background_general["a"].to_numpy()

alpha = params_general.alpha
beta = params_general.beta
omega = params_general.omega

rho0 = params_general.rho0
a0 = params_general.a0

In [17]:
rho_a_residual = (
    np.log(rho_general / rho0)
    + (
        alpha
        * beta
        / (beta - 1.0)
    )
    * (
        rho_general**(beta - 1.0)
        - rho0**(beta - 1.0)
    )
    + 3.0
    * (1.0 + omega)
    * np.log(a_general / a0)
)

print(
    "Máximo residuo absoluto en rho(a):",
    np.max(np.abs(rho_a_residual)),
)

Máximo residuo absoluto en rho(a): 6.661338147750939e-16


In [18]:
rho_a_scale = (
    1.0
    + np.abs(
        np.log(rho_general / rho0)
    )
    + np.abs(
        (
            alpha
            * beta
            / (beta - 1.0)
        )
        * (
            rho_general**(beta - 1.0)
            - rho0**(beta - 1.0)
        )
    )
    + np.abs(
        3.0
        * (1.0 + omega)
        * np.log(a_general / a0)
    )
)

rho_a_normalized_residual = (
    np.abs(rho_a_residual)
    / rho_a_scale
)

print(
    "Máximo residuo normalizado en rho(a):",
    np.max(rho_a_normalized_residual),
)

Máximo residuo normalizado en rho(a): 4.16452175254622e-16


In [19]:
H_general = background_general["H"].to_numpy()

H_expected_general = np.sqrt(
    (
        rho_general
        + alpha * rho_general**beta
    )
    / 3.0
)

friedmann_error_general = np.max(
    np.abs(
        H_general
        - H_expected_general
    )
    / np.maximum(
        np.abs(H_expected_general),
        np.finfo(float).tiny,
    )
)

print(
    "Error relativo máximo en H:",
    friedmann_error_general,
)

Error relativo máximo en H: 1.6862295144896293e-16


In [20]:
Hdot_general = (
    background_general["Hdot"].to_numpy()
)

Hdot_expected_general = (
    -0.5
    * (1.0 + omega)
    * rho_general
)

Hdot_error_general = np.max(
    np.abs(
        Hdot_general
        - Hdot_expected_general
    )
    / np.maximum(
        np.abs(Hdot_expected_general),
        np.finfo(float).tiny,
    )
)

print(
    "Error relativo máximo en Hdot:",
    Hdot_error_general,
)

Error relativo máximo en Hdot: 0.0


In [21]:
t_general = background_general["t"].to_numpy()

rho_dot_numerical = np.gradient(
    rho_general,
    t_general,
    edge_order=2,
)

denominator_general = (
    1.0
    + alpha
    * beta
    * rho_general**(beta - 1.0)
)

rho_dot_expected = (
    -3.0
    * H_general
    * (1.0 + omega)
    * rho_general
    / denominator_general
)

In [22]:
interior = slice(5, -5)

rho_dot_relative_error = np.max(
    np.abs(
        rho_dot_numerical[interior]
        - rho_dot_expected[interior]
    )
    / np.maximum(
        np.abs(
            rho_dot_expected[interior]
        ),
        1e-12,
    )
)

print(
    "Error relativo máximo en dot(rho):",
    rho_dot_relative_error,
)

Error relativo máximo en dot(rho): 2.391113841075464e-09


In [23]:
a_dot_numerical = np.gradient(
    a_general,
    t_general,
    edge_order=2,
)

a_dot_expected = (
    a_general
    * H_general
)

a_dot_relative_error = np.max(
    np.abs(
        a_dot_numerical[interior]
        - a_dot_expected[interior]
    )
    / np.maximum(
        np.abs(
            a_dot_expected[interior]
        ),
        1e-12,
    )
)

print(
    "Error relativo máximo en dot(a):",
    a_dot_relative_error,
)

Error relativo máximo en dot(a): 1.0958355253853794e-08


In [24]:
general_validation = {
    "condiciones_iniciales": all(
        general_initial_checks.values()
    ),
    "relacion_rho_a": (
        np.max(
            rho_a_normalized_residual
        )
        < 1e-7
    ),
    "ecuacion_friedmann": (
        friedmann_error_general
        < 1e-10
    ),
    "ecuacion_Hdot": (
        Hdot_error_general
        < 1e-10
    ),
    "ecuacion_rho_dot": (
        rho_dot_relative_error
        < 1e-3
    ),
    "ecuacion_a_dot": (
        a_dot_relative_error
        < 1e-3
    ),
}

general_validation

{'condiciones_iniciales': True,
 'relacion_rho_a': np.True_,
 'ecuacion_friedmann': np.True_,
 'ecuacion_Hdot': np.True_,
 'ecuacion_rho_dot': np.True_,
 'ecuacion_a_dot': np.True_}

In [25]:
print(
    "Caso beta != 1 validado:",
    all(general_validation.values()),
)

Caso beta != 1 validado: True


## 4. Validación de eventos de detención

Antes de clasificar singularidades debemos distinguir entre:

- `rho_min`: la densidad alcanzó el umbral inferior numérico;
- `rho_max`: la densidad alcanzó el umbral superior numérico;
- `a_max`: el factor de escala alcanzó el umbral superior;
- `effective_density_boundary`:
  \[
  1+\alpha\rho^{\beta-1}\rightarrow0,
  \]
  por lo que \(H^2\rightarrow0\);
- `rho_dot_denominator_boundary`:
  \[
  1+\alpha\beta\rho^{\beta-1}\rightarrow0,
  \]
  donde la ecuación para \(\dot\rho\) se vuelve singular.

Las siguientes pruebas están preparadas para ejecutarse después de confirmar
que la versión actual de `solve_background_t` contiene los cinco eventos.


In [26]:
def run_event_test(
    name,
    params,
    expected_event,
    **solver_kwargs,
):
    """Ejecuta un caso controlado y resume la causa de detención."""

    model = BarotropicPowerDiffusionModel(params)
    background = model.solve_background_t(**solver_kwargs)

    detected_event = background.attrs.get("termination_event")

    return {
        "caso": name,
        "evento_esperado": expected_event,
        "evento_detectado": detected_event,
        "evento_correcto": detected_event == expected_event,
        "tiempo_termino": background.attrs.get("termination_time"),
        "rho_final": background["rho"].iloc[-1],
        "a_final": background["a"].iloc[-1],
        "H_final": background["H"].iloc[-1],
        "Hdot_final": background["Hdot"].iloc[-1],
    }


In [27]:
event_tests = []

# 1. Umbral superior de densidad
event_tests.append(
    run_event_test(
        name="rho_max",
        params=BarotropicPowerDiffusionParams(
            omega=-1.2,
            alpha=0.3,
            beta=1.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        expected_event="rho_max",
        t_end=2.0,
        n_points=500,
        rho_max=1.2,
        a_max=1e12,
    )
)

# 2. Umbral superior del factor de escala
event_tests.append(
    run_event_test(
        name="a_max",
        params=BarotropicPowerDiffusionParams(
            omega=-1.2,
            alpha=0.3,
            beta=1.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        expected_event="a_max",
        t_end=2.0,
        n_points=500,
        rho_max=1e12,
        a_max=1.05,
    )
)

# 3. Umbral inferior de densidad
event_tests.append(
    run_event_test(
        name="rho_min",
        params=BarotropicPowerDiffusionParams(
            omega=0.0,
            alpha=0.3,
            beta=1.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        expected_event="rho_min",
        t_end=2.0,
        n_points=500,
        rho_min=0.8,
        rho_max=1e12,
        a_max=1e12,
    )
)

# 4. Frontera de H^2. Para beta<1 y alpha<0,
#    la evolución no phantom puede acercarse a rho + Q = 0.
event_tests.append(
    run_event_test(
        name="effective_density_boundary",
        params=BarotropicPowerDiffusionParams(
            omega=0.0,
            alpha=-0.5,
            beta=0.5,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        expected_event="effective_density_boundary",
        t_end=20.0,
        n_points=1000,
        rho_min=1e-8,
        rho_max=1e12,
        a_max=1e12,
        boundary_tol=1e-6,
        max_step=0.01,
    )
)

# 5. Frontera del denominador de dot(rho). Para beta>1,
#    alpha<0 y omega<-1, la densidad creciente puede alcanzarla.
event_tests.append(
    run_event_test(
        name="rho_dot_denominator_boundary",
        params=BarotropicPowerDiffusionParams(
            omega=-1.2,
            alpha=-0.3,
            beta=1.5,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        expected_event="rho_dot_denominator_boundary",
        t_end=20.0,
        n_points=1000,
        rho_min=1e-12,
        rho_max=1e12,
        a_max=1e12,
        boundary_tol=1e-6,
        max_step=0.01,
    )
)

event_validation_table = pd.DataFrame(event_tests)
event_validation_table


,caso,evento_esperado,evento_detectado,evento_correcto,tiempo_termino,rho_final,a_final,H_final,Hdot_final
0,rho_max,rho_max,rho_max,True,0.573554,1.200000,1.484429,0.721110,0.120000
1,a_max,a_max,a_max,True,0.073702,1.022774,1.050000,0.665734,0.102277
2,rho_min,rho_min,rho_min,True,0.155399,0.800000,1.101525,0.588784,-0.400000
3,effective_density_boundary,effective_density_boundary,effective_density_boundary,True,1.902849,0.250001,1.343706,0.000289,-0.125000
4,rho_dot_denominator_boundary,rho_dot_denominator_boundary,rho_dot_denominator_boundary,True,1.420159,4.938262,2.289558,0.740741,0.493826


In [28]:
print(
    "Todos los eventos fueron detectados correctamente:",
    bool(event_validation_table["evento_correcto"].all()),
)


Todos los eventos fueron detectados correctamente: True


## 5. Preparación del estudio de intervalos

La clasificación numérica no debe preceder al análisis del dominio.
Para cada combinación de parámetros se estudiarán:

\[
F(\rho)=1+\alpha\rho^{\beta-1}>0,
\]

\[
D(\rho)=1+\alpha\beta\rho^{\beta-1}\neq0,
\]

\[
\dot\rho=
-\frac{3H(1+\omega)\rho}{D(\rho)}.
\]

El objetivo será determinar, para cada intervalo de
\((\alpha,\beta,\omega)\), si la evolución futura conduce a:

\[
\rho\rightarrow0,\qquad
\rho\rightarrow\rho_c<\infty,\qquad
\rho\rightarrow\infty,
\]

y si el límite se alcanza en tiempo finito o infinito.


In [29]:
def critical_density_summary(alpha, beta):
    """Calcula densidades críticas positivas cuando existen."""

    if np.isclose(beta, 1.0):
        return {
            "rho_H_boundary": np.nan,
            "rho_denominator_boundary": np.nan,
            "nota": (
                "Para beta=1, F=1+alpha y "
                "D=1+alpha son constantes."
            ),
        }

    rho_H_boundary = np.nan
    rho_denominator_boundary = np.nan

    # F(rho)=0 requiere alpha<0.
    if alpha < 0:
        rho_H_boundary = (
            -1.0 / alpha
        ) ** (1.0 / (beta - 1.0))

    # D(rho)=0 requiere alpha*beta<0.
    if alpha * beta < 0:
        rho_denominator_boundary = (
            -1.0 / (alpha * beta)
        ) ** (1.0 / (beta - 1.0))

    return {
        "rho_H_boundary": rho_H_boundary,
        "rho_denominator_boundary": rho_denominator_boundary,
        "nota": "",
    }


def evaluate_domain_factors(params, rho_values):
    """Evalúa F(rho), D(rho) y el signo de dot(rho)."""

    rho_values = np.asarray(rho_values, dtype=float)

    F = (
        1.0
        + params.alpha
        * rho_values ** (params.beta - 1.0)
    )

    D = (
        1.0
        + params.alpha
        * params.beta
        * rho_values ** (params.beta - 1.0)
    )

    H = np.full_like(rho_values, np.nan)
    physical = F > 0

    H[physical] = np.sqrt(
        rho_values[physical]
        * F[physical]
        / 3.0
    )

    rho_dot = np.full_like(rho_values, np.nan)
    regular = physical & (~np.isclose(D, 0.0))

    rho_dot[regular] = (
        -3.0
        * H[regular]
        * (1.0 + params.omega)
        * rho_values[regular]
        / D[regular]
    )

    return pd.DataFrame(
        {
            "rho": rho_values,
            "F": F,
            "D": D,
            "H": H,
            "rho_dot": rho_dot,
            "physical": physical,
            "regular": regular,
        }
    )


In [30]:
representative_cases = [
    {
        "case_id": "B1",
        "branch": "beta = 1, fluido no phantom",
        "params": BarotropicPowerDiffusionParams(
            omega=0.0,
            alpha=0.3,
            beta=1.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "rho -> 0, H -> 0, sin singularidad finita",
    },
    {
        "case_id": "B2",
        "branch": "beta = 1, omega = -1",
        "params": BarotropicPowerDiffusionParams(
            omega=-1.0,
            alpha=0.3,
            beta=1.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "rho y H constantes; expansion de Sitter",
    },
    {
        "case_id": "B3",
        "branch": "beta = 1, fluido phantom",
        "params": BarotropicPowerDiffusionParams(
            omega=-1.2,
            alpha=0.3,
            beta=1.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "Big Rip en tiempo finito",
    },
    {
        "case_id": "C1",
        "branch": "1 < beta < 2, alpha > 0, phantom",
        "params": BarotropicPowerDiffusionParams(
            omega=-1.2,
            alpha=0.3,
            beta=1.5,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "rho, a y H divergen en tiempo finito",
    },
    {
        "case_id": "C2",
        "branch": "beta = 2, alpha > 0, phantom",
        "params": BarotropicPowerDiffusionParams(
            omega=-1.2,
            alpha=0.3,
            beta=2.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "divergencia solamente cuando t -> infinito",
    },
    {
        "case_id": "C3",
        "branch": "beta > 2, alpha > 0, phantom",
        "params": BarotropicPowerDiffusionParams(
            omega=-1.2,
            alpha=0.3,
            beta=3.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "divergencia solamente cuando t -> infinito",
    },
    {
        "case_id": "C4",
        "branch": "beta > 1, alpha < 0, phantom",
        "params": BarotropicPowerDiffusionParams(
            omega=-1.2,
            alpha=-0.3,
            beta=1.5,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "evolucion hacia la frontera D(rho) = 0",
    },
    {
        "case_id": "D1",
        "branch": "0 < beta < 1, alpha > 0, no phantom",
        "params": BarotropicPowerDiffusionParams(
            omega=0.0,
            alpha=0.3,
            beta=0.5,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "rho -> 0 y H -> 0 en tiempo infinito",
    },
    {
        "case_id": "D2",
        "branch": "0 < beta < 1, alpha > 0, phantom",
        "params": BarotropicPowerDiffusionParams(
            omega=-1.2,
            alpha=0.3,
            beta=0.5,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "Big Rip en tiempo finito",
    },
    {
        "case_id": "D3",
        "branch": "0 < beta < 1, alpha < 0, no phantom",
        "params": BarotropicPowerDiffusionParams(
            omega=0.0,
            alpha=-0.5,
            beta=0.5,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "rho -> rho_H y H -> 0",
    },
    {
        "case_id": "E1",
        "branch": "beta = 0, alpha > 0, no phantom",
        "params": BarotropicPowerDiffusionParams(
            omega=0.0,
            alpha=0.3,
            beta=0.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "rho -> 0 y H -> sqrt(alpha/3)",
    },
    {
        "case_id": "E2",
        "branch": "beta = 0, alpha < 0, no phantom",
        "params": BarotropicPowerDiffusionParams(
            omega=0.0,
            alpha=-0.5,
            beta=0.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "rho -> -alpha y H -> 0",
    },
    {
        "case_id": "F1",
        "branch": "beta < 0, alpha > 0, phantom, rho0 > rho_D",
        "params": BarotropicPowerDiffusionParams(
            omega=-1.2,
            alpha=0.25,
            beta=-1.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "rho crece hacia infinito",
    },
    {
        "case_id": "F2",
        "branch": "beta < 0, alpha > 0, phantom, rho0 < rho_D",
        "params": BarotropicPowerDiffusionParams(
            omega=-1.2,
            alpha=4.0,
            beta=-1.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "rho -> 0, H -> infinito y Hdot -> 0",
    },
    {
        "case_id": "F3",
        "branch": "beta < 0, alpha > 0, no phantom",
        "params": BarotropicPowerDiffusionParams(
            omega=0.0,
            alpha=0.25,
            beta=-1.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "evolucion hacia la frontera D(rho) = 0",
    },
    {
        "case_id": "F4",
        "branch": "beta < 0, alpha < 0, no phantom",
        "params": BarotropicPowerDiffusionParams(
            omega=0.0,
            alpha=-0.5,
            beta=-1.0,
            rho0=1.0,
            a0=1.0,
            t0=0.0,
        ),
        "expected_behavior": "rho -> rho_H y H -> 0",
    },
]

In [31]:
def summarize_representative_case(case):
    params = case["params"]

    rho0 = params.rho0

    F0 = (
        1.0
        + params.alpha
        * rho0 ** (params.beta - 1.0)
    )

    D0 = (
        1.0
        + params.alpha
        * params.beta
        * rho0 ** (params.beta - 1.0)
    )

    critical = critical_density_summary(
        alpha=params.alpha,
        beta=params.beta,
    )

    if np.isclose(1.0 + params.omega, 0.0):
        initial_direction = "constante"
    elif D0 > 0:
        initial_direction = (
            "creciente"
            if params.omega < -1.0
            else "decreciente"
        )
    else:
        initial_direction = (
            "decreciente"
            if params.omega < -1.0
            else "creciente"
        )

    return {
        "case_id": case["case_id"],
        "branch": case["branch"],
        "omega": params.omega,
        "alpha": params.alpha,
        "beta": params.beta,
        "rho0": rho0,
        "F0": F0,
        "D0": D0,
        "physical_initial_state": F0 > 0,
        "regular_initial_state": (
            F0 > 0
            and not np.isclose(D0, 0.0)
        ),
        "initial_rho_direction": initial_direction,
        "rho_H": critical["rho_H_boundary"],
        "rho_D": critical["rho_denominator_boundary"],
        "expected_behavior": case["expected_behavior"],
    }

In [32]:
representative_case_table = pd.DataFrame(
    [
        summarize_representative_case(case)
        for case in representative_cases
    ]
)

representative_case_table

,case_id,branch,omega,alpha,beta,rho0,F0,D0,physical_initial_state,regular_initial_state,initial_rho_direction,rho_H,rho_D,expected_behavior
0,B1,"beta = 1, fluido no phantom",0.0,0.30,1.0,1.0,1.30,1.30,True,True,decreciente,NaN,NaN,"rho -> 0, H -> 0, sin singularidad finita"
1,B2,"beta = 1, omega = -1",-1.0,0.30,1.0,1.0,1.30,1.30,True,True,constante,NaN,NaN,rho y H constantes; expansion de Sitter
2,B3,"beta = 1, fluido phantom",-1.2,0.30,1.0,1.0,1.30,1.30,True,True,creciente,NaN,NaN,Big Rip en tiempo finito
3,C1,"1 < beta < 2, alpha > 0, phantom",-1.2,0.30,1.5,1.0,1.30,1.45,True,True,creciente,NaN,NaN,"rho, a y H divergen en tiempo finito"
4,C2,"beta = 2, alpha > 0, phantom",-1.2,0.30,2.0,1.0,1.30,1.60,True,True,creciente,NaN,NaN,divergencia solamente cuando t -> infinito
5,C3,"beta > 2, alpha > 0, phantom",-1.2,0.30,3.0,1.0,1.30,1.90,True,True,creciente,NaN,NaN,divergencia solamente cuando t -> infinito
6,C4,"beta > 1, alpha < 0, phantom",-1.2,-0.30,1.5,1.0,0.70,0.55,True,True,creciente,11.111111,4.938272,evolucion hacia la frontera D(rho) = 0
7,D1,"0 < beta < 1, alpha > 0, no phantom",0.0,0.30,0.5,1.0,1.30,1.15,True,True,decreciente,NaN,NaN,rho -> 0 y H -> 0 en tiempo infinito
8,D2,"0 < beta < 1, alpha > 0, phantom",-1.2,0.30,0.5,1.0,1.30,1.15,True,True,creciente,NaN,NaN,Big Rip en tiempo finito
9,D3,"0 < beta < 1, alpha < 0, no phantom",0.0,-0.50,0.5,1.0,0.50,0.75,True,True,decreciente,0.250000,0.062500,rho -> rho_H y H -> 0


In [33]:
representative_case_checks = {
    "todos_fisicos_inicialmente": bool(
        representative_case_table[
            "physical_initial_state"
        ].all()
    ),
    "todos_regulares_inicialmente": bool(
        representative_case_table[
            "regular_initial_state"
        ].all()
    ),
    "identificadores_unicos": bool(
        representative_case_table[
            "case_id"
        ].is_unique
    ),
}

representative_case_checks

{'todos_fisicos_inicialmente': True,
 'todos_regulares_inicialmente': True,
 'identificadores_unicos': True}

In [34]:
representative_case_table[
    [
        "case_id",
        "rho0",
        "rho_H",
        "rho_D",
        "initial_rho_direction",
    ]
]

,case_id,rho0,rho_H,rho_D,initial_rho_direction
0,B1,1.0,NaN,NaN,decreciente
1,B2,1.0,NaN,NaN,constante
2,B3,1.0,NaN,NaN,creciente
3,C1,1.0,NaN,NaN,creciente
4,C2,1.0,NaN,NaN,creciente
5,C3,1.0,NaN,NaN,creciente
6,C4,1.0,11.111111,4.938272,creciente
7,D1,1.0,NaN,NaN,decreciente
8,D2,1.0,NaN,NaN,creciente
9,D3,1.0,0.250000,0.062500,decreciente


### Pendiente: tabla definitiva de intervalos

La siguiente etapa del proyecto será completar una tabla con columnas como:

| Intervalo de parámetros | Dominio físico | Signo de \(\dot\rho\) | Límite futuro de \(\rho\) | Tiempo finito/infinito | Evento relevante |
|---|---|---|---|---|---|

Esta tabla se derivará matemáticamente antes de fijar los casos que alimentarán
el clasificador.


## 6. Clasificador de singularidades — pendiente

El clasificador se implementará después del estudio de intervalos.

Usará exclusivamente las evoluciones de

\[
\rho(t),\qquad a(t),\qquad H(t),\qquad \dot H(t),
\]

junto con:

- el tiempo final alcanzado;
- el evento de detención;
- las tendencias del tramo final;
- los límites analíticos obtenidos para cada intervalo.

Recordatorio: el **Big Rip corresponde a una singularidad tipo I**.

La frontera \(D(\rho)=0\) requiere atención especial. Aunque
\(\rho\), \(a\), \(H\) y \(\dot H\) puedan mantenerse finitos,

\[
\ddot H=-\frac{1+\omega}{2}\dot\rho
\]

puede divergir. Esta información puede inferirse analíticamente sin añadir
\(\ddot H(t)\) como salida del solucionador.


In [35]:
# TODO: implementar después del estudio de intervalos.
#
# Esquema previsto:
#
# classification = classify_barotropic_power_diffusion(
#     background=background,
#     params=params,
# )
#
# La función se incorporará en src/classifiers.py cuando las reglas
# matemáticas estén completamente definidas.

default_integration_config = {
    "t_end": 50.0,
    "n_points": 1500,
    "rho_min": 1e-12,
    "rho_max": 1e12,
    "a_max": 1e12,
    "boundary_tol": 1e-8,
    "method": "DOP853",
    "rtol": 1e-9,
    "atol": 1e-11,
    "max_step": 0.05,
}

In [36]:
case_integration_config = {
    "B1": {
        "t_end": 50.0,
        "max_step": 0.1,
    },
    "B2": {
        "t_end": 20.0,
        "max_step": 0.1,
    },
    "B3": {
        "t_end": 10.0,
        "a_max": 1e10,
        "max_step": 0.01,
    },
    "C1": {
        "t_end": 50.0,
        "a_max": 1e10,
        "rho_max": 1e10,
        "max_step": 0.01,
    },
    "C2": {
        "t_end": 100.0,
        "a_max": 1e20,
        "rho_max": 1e12,
        "max_step": 0.05,
    },
    "C3": {
        "t_end": 200.0,
        "a_max": 1e20,
        "rho_max": 1e12,
        "max_step": 0.1,
    },
    "C4": {
        "t_end": 20.0,
        "max_step": 0.005,
    },
    "D1": {
        "t_end": 200.0,
        "max_step": 0.1,
    },
    "D2": {
        "t_end": 20.0,
        "a_max": 1e10,
        "max_step": 0.01,
    },
    "D3": {
        "t_end": 20.0,
        "max_step": 0.01,
    },
    "E1": {
        "t_end": 50.0,
        "max_step": 0.05,
    },
    "E2": {
        "t_end": 20.0,
        "max_step": 0.01,
    },
    "F1": {
        "t_end": 20.0,
        "a_max": 1e10,
        "max_step": 0.01,
    },
    "F2": {
        "t_end": 500.0,
        "a_max": 1e100,
        "rho_min": 1e-10,
        "max_step": 0.2,
    },
    "F3": {
        "t_end": 20.0,
        "max_step": 0.005,
    },
    "F4": {
        "t_end": 20.0,
        "max_step": 0.01,
    },
}

In [37]:
def integrate_representative_case(case):
    """
    Integra un caso representativo y devuelve:
    - el modelo;
    - la evolución temporal;
    - la configuración utilizada.
    """

    case_id = case["case_id"]
    params = case["params"]

    solver_config = default_integration_config.copy()

    solver_config.update(
        case_integration_config.get(
            case_id,
            {},
        )
    )

    model = BarotropicPowerDiffusionModel(params)

    background = model.solve_background_t(
        **solver_config
    )

    return {
        "case_id": case_id,
        "model": model,
        "background": background,
        "solver_config": solver_config,
    }

In [38]:
integrated_cases = {}

integration_errors = {}

for case in representative_cases:
    case_id = case["case_id"]

    try:
        result = integrate_representative_case(
            case
        )

        integrated_cases[case_id] = result

        print(
            f"{case_id}: integración completada"
        )

    except Exception as exc:
        integration_errors[case_id] = str(exc)

        print(
            f"{case_id}: error -> {exc}"
        )

B1: integración completada
B2: integración completada
B3: integración completada
C1: integración completada
C2: integración completada
C3: integración completada
C4: integración completada
D1: integración completada
D2: integración completada
D3: integración completada
E1: integración completada
E2: integración completada
F1: integración completada
F2: integración completada
F3: error -> El integrador no logró completar la evolución: Required step size is less than spacing between numbers.
F4: integración completada


In [39]:
integration_errors

{'F3': 'El integrador no logró completar la evolución: Required step size is less than spacing between numbers.'}

In [40]:
print(
    "Casos integrados:",
    len(integrated_cases),
)

print(
    "Casos esperados:",
    len(representative_cases),
)

Casos integrados: 15
Casos esperados: 16


## 7. Gráficos finales — pendiente

Después de validar el clasificador se construirán gráficos de:

\[
\rho(t),\qquad a(t),\qquad H(t),\qquad \dot H(t),
\]

separados por los intervalos físicos de \(\alpha\), \(\beta\) y \(\omega\).
No se fijan todavía los casos representativos para evitar elegir parámetros
antes de completar el análisis matemático.


In [41]:
# TODO: generar los gráficos finales una vez definidos
# los intervalos y validado el clasificador.
#
# Variables que deberán representarse:
# - background["rho"]
# - background["a"]
# - background["H"]
# - background["Hdot"]

def summarize_integrated_case(case, result):
    params = case["params"]
    background = result["background"]

    initial = background.iloc[0]
    final = background.iloc[-1]

    termination_event = background.attrs.get(
        "termination_event"
    )

    if termination_event is None:
        termination_event = "t_end"

    rho_final = float(final["rho"])

    F_final = (
        1.0
        + params.alpha
        * rho_final ** (params.beta - 1.0)
    )

    D_final = (
        1.0
        + params.alpha
        * params.beta
        * rho_final ** (params.beta - 1.0)
    )

    critical = critical_density_summary(
        alpha=params.alpha,
        beta=params.beta,
    )

    return {
        "case_id": case["case_id"],
        "branch": case["branch"],
        "omega": params.omega,
        "alpha": params.alpha,
        "beta": params.beta,
        "rho0": params.rho0,
        "a0": params.a0,
        "H0": float(initial["H"]),
        "Hdot0": float(initial["Hdot"]),
        "t_final": float(final["t"]),
        "rho_final": rho_final,
        "a_final": float(final["a"]),
        "H_final": float(final["H"]),
        "Hdot_final": float(final["Hdot"]),
        "F_final": float(F_final),
        "D_final": float(D_final),
        "rho_H": critical[
            "rho_H_boundary"
        ],
        "rho_D": critical[
            "rho_denominator_boundary"
        ],
        "termination_event": termination_event,
        "solver_success": bool(
            background.attrs.get(
                "solver_success",
                False,
            )
        ),
        "reached_t_end": bool(
            background.attrs.get(
                "reached_t_end",
                False,
            )
        ),
        "expected_behavior": case[
            "expected_behavior"
        ],
    }

In [43]:
integration_summary = pd.DataFrame(
    [
        summarize_integrated_case(
            case,
            integrated_cases[case["case_id"]],
        )
        for case in representative_cases
        if case["case_id"] in integrated_cases
    ]
)

integration_summary

,case_id,branch,omega,alpha,beta,rho0,a0,H0,Hdot0,t_final,...,H_final,Hdot_final,F_final,D_final,rho_H,rho_D,termination_event,solver_success,reached_t_end,expected_behavior
0,B1,"beta = 1, fluido no phantom",0.0,0.30,1.0,1.0,1.0,0.658281,-0.5,50.000000,...,0.016889,-3.291069e-04,1.300000e+00,1.300000e+00,NaN,NaN,t_end,True,True,"rho -> 0, H -> 0, sin singularidad finita"
1,B2,"beta = 1, omega = -1",-1.0,0.30,1.0,1.0,1.0,0.658281,-0.0,20.000000,...,0.658281,-0.000000e+00,1.300000e+00,1.300000e+00,NaN,NaN,t_end,True,True,rho y H constantes; expansion de Sitter
2,B3,"beta = 1, fluido phantom",-1.2,0.30,1.0,1.0,1.0,0.658281,0.1,6.550393,...,133.691365,4.124626e+03,1.300000e+00,1.300000e+00,NaN,NaN,a_max,True,False,Big Rip en tiempo finito
3,C1,"1 < beta < 2, alpha > 0, phantom",-1.2,0.30,1.5,1.0,1.0,0.658281,0.1,8.396272,...,13.199622,1.214014e+01,4.305469e+00,5.958204e+00,NaN,NaN,a_max,True,False,"rho, a y H divergen en tiempo finito"
4,C2,"beta = 2, alpha > 0, phantom",-1.2,0.30,2.0,1.0,1.0,0.658281,0.1,12.418162,...,13.440249,4.086780e+00,1.326034e+01,2.552068e+01,NaN,NaN,a_max,True,False,divergencia solamente cuando t -> infinito
5,C3,"beta > 2, alpha > 0, phantom",-1.2,0.30,3.0,1.0,1.0,0.658281,0.1,16.173331,...,6.825338,7.608727e-01,1.836782e+01,5.310345e+01,NaN,NaN,a_max,True,False,divergencia solamente cuando t -> infinito
6,C4,"beta > 1, alpha < 0, phantom",-1.2,-0.30,1.5,1.0,1.0,0.483046,0.1,1.420159,...,0.740741,4.938272e-01,3.333333e-01,-3.071056e-09,11.111111,4.938272,rho_dot_denominator_boundary,True,False,evolucion hacia la frontera D(rho) = 0
7,D1,"0 < beta < 1, alpha > 0, no phantom",0.0,0.30,0.5,1.0,1.0,0.658281,-0.5,200.000000,...,0.034206,-6.358241e-05,2.760344e+01,1.430172e+01,NaN,NaN,t_end,True,True,rho -> 0 y H -> 0 en tiempo infinito
8,D2,"0 < beta < 1, alpha > 0, phantom",-1.2,0.30,0.5,1.0,1.0,0.658281,0.1,5.784465,...,497.103172,7.410764e+04,1.000348e+00,1.000174e+00,NaN,NaN,a_max,True,False,Big Rip en tiempo finito
9,D3,"0 < beta < 1, alpha < 0, no phantom",0.0,-0.50,0.5,1.0,1.0,0.408248,-0.5,1.904928,...,0.000029,-1.250000e-01,1.000000e-08,5.000000e-01,0.250000,0.062500,effective_density_boundary,True,False,rho -> rho_H y H -> 0


In [44]:
initial_cosmological_parameters = (
    integration_summary[
        [
            "case_id",
            "omega",
            "alpha",
            "beta",
            "rho0",
            "a0",
            "H0",
            "Hdot0",
        ]
    ]
    .copy()
)

initial_cosmological_parameters

,case_id,omega,alpha,beta,rho0,a0,H0,Hdot0
0,B1,0.0,0.30,1.0,1.0,1.0,0.658281,-0.5
1,B2,-1.0,0.30,1.0,1.0,1.0,0.658281,-0.0
2,B3,-1.2,0.30,1.0,1.0,1.0,0.658281,0.1
3,C1,-1.2,0.30,1.5,1.0,1.0,0.658281,0.1
4,C2,-1.2,0.30,2.0,1.0,1.0,0.658281,0.1
5,C3,-1.2,0.30,3.0,1.0,1.0,0.658281,0.1
6,C4,-1.2,-0.30,1.5,1.0,1.0,0.483046,0.1
7,D1,0.0,0.30,0.5,1.0,1.0,0.658281,-0.5
8,D2,-1.2,0.30,0.5,1.0,1.0,0.658281,0.1
9,D3,0.0,-0.50,0.5,1.0,1.0,0.408248,-0.5


In [45]:
final_cosmological_values = (
    integration_summary[
        [
            "case_id",
            "t_final",
            "rho_final",
            "a_final",
            "H_final",
            "Hdot_final",
            "F_final",
            "D_final",
            "termination_event",
            "solver_success",
            "reached_t_end",
        ]
    ]
    .copy()
)

final_cosmological_values

,case_id,t_final,rho_final,a_final,H_final,Hdot_final,F_final,D_final,termination_event,solver_success,reached_t_end
0,B1,50.000000,6.582138e-04,2.391705e+01,0.016889,-3.291069e-04,1.300000e+00,1.300000e+00,t_end,True,True
1,B2,20.000000,1.000000e+00,5.220986e+05,0.658281,-0.000000e+00,1.300000e+00,1.300000e+00,t_end,True,True
2,B3,6.550393,4.124626e+04,1.000000e+10,133.691365,4.124626e+03,1.300000e+00,1.300000e+00,a_max,True,False
3,C1,8.396272,1.214014e+02,1.000000e+10,13.199622,1.214014e+01,4.305469e+00,5.958204e+00,a_max,True,False
4,C2,12.418162,4.086780e+01,1.000000e+20,13.440249,4.086780e+00,1.326034e+01,2.552068e+01,a_max,True,False
5,C3,16.173331,7.608727e+00,1.000000e+20,6.825338,7.608727e-01,1.836782e+01,5.310345e+01,a_max,True,False
6,C4,1.420159,4.938272e+00,2.289558e+00,0.740741,4.938272e-01,3.333333e-01,-3.071056e-09,rho_dot_denominator_boundary,True,False
7,D1,200.000000,1.271648e-04,1.277489e+05,0.034206,-6.358241e-05,2.760344e+01,1.430172e+01,t_end,True,True
8,D2,5.784465,7.410764e+05,1.000000e+10,497.103172,7.410764e+04,1.000348e+00,1.000174e+00,a_max,True,False
9,D3,1.904928,2.500000e-01,1.343706e+00,0.000029,-1.250000e-01,1.000000e-08,5.000000e-01,effective_density_boundary,True,False


In [46]:
with pd.option_context(
    "display.float_format",
    "{:.6e}".format,
):
    display(final_cosmological_values)

,case_id,t_final,rho_final,a_final,H_final,Hdot_final,F_final,D_final,termination_event,solver_success,reached_t_end
0,B1,5.000000e+01,6.582138e-04,2.391705e+01,1.688863e-02,-3.291069e-04,1.300000e+00,1.300000e+00,t_end,True,True
1,B2,2.000000e+01,1.000000e+00,5.220986e+05,6.582806e-01,-0.000000e+00,1.300000e+00,1.300000e+00,t_end,True,True
2,B3,6.550393e+00,4.124626e+04,1.000000e+10,1.336914e+02,4.124626e+03,1.300000e+00,1.300000e+00,a_max,True,False
3,C1,8.396272e+00,1.214014e+02,1.000000e+10,1.319962e+01,1.214014e+01,4.305469e+00,5.958204e+00,a_max,True,False
4,C2,1.241816e+01,4.086780e+01,1.000000e+20,1.344025e+01,4.086780e+00,1.326034e+01,2.552068e+01,a_max,True,False
5,C3,1.617333e+01,7.608727e+00,1.000000e+20,6.825338e+00,7.608727e-01,1.836782e+01,5.310345e+01,a_max,True,False
6,C4,1.420159e+00,4.938272e+00,2.289558e+00,7.407407e-01,4.938272e-01,3.333333e-01,-3.071056e-09,rho_dot_denominator_boundary,True,False
7,D1,2.000000e+02,1.271648e-04,1.277489e+05,3.420617e-02,-6.358241e-05,2.760344e+01,1.430172e+01,t_end,True,True
8,D2,5.784465e+00,7.410764e+05,1.000000e+10,4.971032e+02,7.410764e+04,1.000348e+00,1.000174e+00,a_max,True,False
9,D3,1.904928e+00,2.500000e-01,1.343706e+00,2.886751e-05,-1.250000e-01,1.000000e-08,5.000000e-01,effective_density_boundary,True,False


In [47]:
integration_checks = {
    "todos_los_casos_integrados": (
        len(integrated_cases)
        == len(representative_cases)
    ),
    "sin_errores_de_integracion": (
        len(integration_errors) == 0
    ),
    "todos_solver_success": bool(
        integration_summary[
            "solver_success"
        ].all()
    ),
    "todos_valores_finitos": bool(
        np.isfinite(
            integration_summary[
                [
                    "t_final",
                    "rho_final",
                    "a_final",
                    "H_final",
                    "Hdot_final",
                ]
            ].to_numpy()
        ).all()
    ),
}

integration_checks

{'todos_los_casos_integrados': False,
 'sin_errores_de_integracion': False,
 'todos_solver_success': True,
 'todos_valores_finitos': True}

In [48]:
event_summary = (
    integration_summary[
        [
            "case_id",
            "termination_event",
            "t_final",
            "rho_final",
            "a_final",
            "rho_H",
            "rho_D",
            "expected_behavior",
        ]
    ]
    .copy()
)

event_summary

,case_id,termination_event,t_final,rho_final,a_final,rho_H,rho_D,expected_behavior
0,B1,t_end,50.000000,6.582138e-04,2.391705e+01,NaN,NaN,"rho -> 0, H -> 0, sin singularidad finita"
1,B2,t_end,20.000000,1.000000e+00,5.220986e+05,NaN,NaN,rho y H constantes; expansion de Sitter
2,B3,a_max,6.550393,4.124626e+04,1.000000e+10,NaN,NaN,Big Rip en tiempo finito
3,C1,a_max,8.396272,1.214014e+02,1.000000e+10,NaN,NaN,"rho, a y H divergen en tiempo finito"
4,C2,a_max,12.418162,4.086780e+01,1.000000e+20,NaN,NaN,divergencia solamente cuando t -> infinito
5,C3,a_max,16.173331,7.608727e+00,1.000000e+20,NaN,NaN,divergencia solamente cuando t -> infinito
6,C4,rho_dot_denominator_boundary,1.420159,4.938272e+00,2.289558e+00,11.111111,4.938272,evolucion hacia la frontera D(rho) = 0
7,D1,t_end,200.000000,1.271648e-04,1.277489e+05,NaN,NaN,rho -> 0 y H -> 0 en tiempo infinito
8,D2,a_max,5.784465,7.410764e+05,1.000000e+10,NaN,NaN,Big Rip en tiempo finito
9,D3,effective_density_boundary,1.904928,2.500000e-01,1.343706e+00,0.250000,0.062500,rho -> rho_H y H -> 0


In [49]:
background_C1 = integrated_cases[
    "C1"
]["background"]

background_C1.head()

,t,rho,a,H,Hdot
0,0.000000,1.000000,1.000000,0.658281,0.100000
1,0.005601,1.001527,1.003696,0.658841,0.100153
2,0.011202,1.003058,1.007408,0.659403,0.100306
3,0.016804,1.004592,1.011137,0.659965,0.100459
4,0.022405,1.006129,1.014884,0.660528,0.100613


In [50]:
background_C1.tail()

,t,rho,a,H,Hdot
1495,8.373867,117.853587,7.462308e+09,12.931622,11.785359
1496,8.379469,118.728004,8.024371e+09,12.997879,11.872800
1497,8.385070,119.610718,8.631984e+09,13.064628,11.961072
1498,8.390671,120.501823,9.289091e+09,13.131875,12.050182
1499,8.396272,121.401417,1.000000e+10,13.199622,12.140142


In [51]:
background_F2 = integrated_cases[
    "F2"
]["background"]

background_F2.tail()

,t,rho,a,H,Hdot
1495,87.478153,0.118756,4.556790e+99,3.356650,0.011876
1496,87.536667,0.118706,5.545846e+99,3.357345,0.011871
1497,87.595181,0.118657,6.749851e+99,3.358039,0.011866
1498,87.653695,0.118608,8.215581e+99,3.358733,0.011861
1499,87.712208,0.118558,1.000000e+100,3.359427,0.011856


In [52]:
integration_errors
integration_checks
event_summary

,case_id,termination_event,t_final,rho_final,a_final,rho_H,rho_D,expected_behavior
0,B1,t_end,50.000000,6.582138e-04,2.391705e+01,NaN,NaN,"rho -> 0, H -> 0, sin singularidad finita"
1,B2,t_end,20.000000,1.000000e+00,5.220986e+05,NaN,NaN,rho y H constantes; expansion de Sitter
2,B3,a_max,6.550393,4.124626e+04,1.000000e+10,NaN,NaN,Big Rip en tiempo finito
3,C1,a_max,8.396272,1.214014e+02,1.000000e+10,NaN,NaN,"rho, a y H divergen en tiempo finito"
4,C2,a_max,12.418162,4.086780e+01,1.000000e+20,NaN,NaN,divergencia solamente cuando t -> infinito
5,C3,a_max,16.173331,7.608727e+00,1.000000e+20,NaN,NaN,divergencia solamente cuando t -> infinito
6,C4,rho_dot_denominator_boundary,1.420159,4.938272e+00,2.289558e+00,11.111111,4.938272,evolucion hacia la frontera D(rho) = 0
7,D1,t_end,200.000000,1.271648e-04,1.277489e+05,NaN,NaN,rho -> 0 y H -> 0 en tiempo infinito
8,D2,a_max,5.784465,7.410764e+05,1.000000e+10,NaN,NaN,Big Rip en tiempo finito
9,D3,effective_density_boundary,1.904928,2.500000e-01,1.343706e+00,0.250000,0.062500,rho -> rho_H y H -> 0


In [53]:
from scipy.integrate import quad

def state_at_density(params, rho_target):
    """
    Calcula el tiempo cósmico y las variables cosmológicas
    cuando la solución alcanza una densidad rho_target.

    La integración se realiza usando u = ln(rho).
    Esta función solo debe utilizarse dentro de una rama
    monótona que no cruce F(rho)=0 ni D(rho)=0.
    """

    rho_target = float(rho_target)

    if rho_target <= 0.0:
        raise ValueError(
            "rho_target debe ser estrictamente positiva."
        )

    if np.isclose(1.0 + params.omega, 0.0):
        raise ValueError(
            "Para omega = -1 la densidad es constante."
        )

    u0 = np.log(params.rho0)
    u_target = np.log(rho_target)

    # Comprobación del dominio completo entre rho0 y rho_target
    u_grid = np.linspace(
        min(u0, u_target),
        max(u0, u_target),
        2000,
    )

    rho_grid = np.exp(u_grid)

    power_grid = rho_grid ** (
        params.beta - 1.0
    )

    F_grid = (
        1.0
        + params.alpha * power_grid
    )

    D_grid = (
        1.0
        + params.alpha
        * params.beta
        * power_grid
    )

    if np.any(F_grid <= 0.0):
        raise ValueError(
            "La trayectoria cruza una región con H^2 <= 0."
        )

    if np.any(np.abs(D_grid) <= 1e-10):
        raise ValueError(
            "La trayectoria cruza la frontera D(rho)=0."
        )

    if np.any(np.sign(D_grid) != np.sign(D_grid[0])):
        raise ValueError(
            "D(rho) cambia de signo durante la trayectoria."
        )

    def dt_du(u):
        rho = np.exp(u)

        power = rho ** (
            params.beta - 1.0
        )

        F = (
            1.0
            + params.alpha * power
        )

        D = (
            1.0
            + params.alpha
            * params.beta
            * power
        )

        H = np.sqrt(
            rho * F / 3.0
        )

        return (
            -D
            / (
                3.0
                * (1.0 + params.omega)
                * H
            )
        )

    delta_t, quadrature_error = quad(
        dt_du,
        u0,
        u_target,
        epsabs=1e-10,
        epsrel=1e-10,
        limit=500,
    )

    t_target = params.t0 + delta_t

    power_target = rho_target ** (
        params.beta - 1.0
    )

    F_target = (
        1.0
        + params.alpha
        * power_target
    )

    D_target = (
        1.0
        + params.alpha
        * params.beta
        * power_target
    )

    H_target = np.sqrt(
        rho_target * F_target / 3.0
    )

    Hdot_target = (
        -0.5
        * (1.0 + params.omega)
        * rho_target
    )

    # Relación integrada entre rho y a
    if np.isclose(params.beta, 1.0):
        conservation_term = (
            (1.0 + params.alpha)
            * np.log(
                rho_target / params.rho0
            )
        )
    else:
        conservation_term = (
            np.log(
                rho_target / params.rho0
            )
            + (
                params.alpha
                * params.beta
                / (params.beta - 1.0)
            )
            * (
                rho_target ** (
                    params.beta - 1.0
                )
                - params.rho0 ** (
                    params.beta - 1.0
                )
            )
        )

    log_a_target = (
        np.log(params.a0)
        - conservation_term
        / (
            3.0
            * (1.0 + params.omega)
        )
    )

    return {
        "t_target": float(t_target),
        "rho_target": rho_target,
        "log_a_target": float(log_a_target),
        "log10_a_target": float(
            log_a_target / np.log(10.0)
        ),
        "H_target": float(H_target),
        "Hdot_target": float(Hdot_target),
        "F_target": float(F_target),
        "D_target": float(D_target),
        "quadrature_error": float(
            quadrature_error
        ),
    }

In [54]:
case_lookup = {
    case["case_id"]: case
    for case in representative_cases
}

In [55]:
def scan_density_targets(
    case_id,
    density_targets,
):
    """
    Calcula el tiempo necesario para alcanzar cada
    densidad objetivo de un caso representativo.
    """

    if case_id not in case_lookup:
        raise KeyError(
            f"No existe el caso {case_id}."
        )

    case = case_lookup[case_id]
    params = case["params"]

    rows = []

    for rho_target in density_targets:
        state = state_at_density(
            params=params,
            rho_target=rho_target,
        )

        rows.append(
            {
                "case_id": case_id,
                "omega": params.omega,
                "alpha": params.alpha,
                "beta": params.beta,
                **state,
            }
        )

    result = pd.DataFrame(rows)

    result["delta_t"] = (
        result["t_target"].diff()
    )

    result["delta_ratio"] = (
        result["delta_t"]
        / result["delta_t"].shift(1)
    )

    return result

In [57]:
growing_density_cases = [
    "B3",
    "C1",
    "C2",
    "C3",
    "D2",
    "F1",
]

growing_density_targets = [
    1e2,
    1e4,
    1e6,
    1e8,
    1e10,
    1e12,
]

In [58]:
growing_convergence_tables = []

for case_id in growing_density_cases:
    table = scan_density_targets(
        case_id=case_id,
        density_targets=growing_density_targets,
    )

    growing_convergence_tables.append(
        table
    )

growing_convergence = pd.concat(
    growing_convergence_tables,
    ignore_index=True,
)

growing_convergence

,case_id,omega,alpha,beta,t_target,rho_target,log_a_target,log10_a_target,H_target,Hdot_target,F_target,D_target,quadrature_error,delta_t,delta_ratio
0,B3,-1.2,0.30,1.0,5.924525e+00,1.000000e+02,9.977869e+00,4.333333e+00,6.582806e+00,1.000000e+01,1.300000e+00,1.300000e+00,6.577544e-14,NaN,NaN
1,B3,-1.2,0.30,1.0,6.516978e+00,1.000000e+04,1.995574e+01,8.666667e+00,6.582806e+01,1.000000e+03,1.300000e+00,1.300000e+00,7.235299e-14,5.924525e-01,NaN
2,B3,-1.2,0.30,1.0,6.576223e+00,1.000000e+06,2.993361e+01,1.300000e+01,6.582806e+02,1.000000e+05,1.300000e+00,1.300000e+00,7.301074e-14,5.924525e-02,0.100000
3,B3,-1.2,0.30,1.0,6.582148e+00,1.000000e+08,3.991147e+01,1.733333e+01,6.582806e+03,1.000000e+07,1.300000e+00,1.300000e+00,7.307652e-14,5.924525e-03,0.100000
4,B3,-1.2,0.30,1.0,6.582740e+00,1.000000e+10,4.988934e+01,2.166667e+01,6.582806e+04,1.000000e+09,1.300000e+00,1.300000e+00,5.687072e-12,5.924525e-04,0.100000
5,B3,-1.2,0.30,1.0,6.582799e+00,1.000000e+12,5.986721e+01,2.600000e+01,6.582806e+05,1.000000e+11,1.300000e+00,1.300000e+00,3.913918e-10,5.924525e-05,0.100000
6,C1,-1.2,0.30,1.5,8.246388e+00,1.000000e+02,2.117528e+01,9.196309e+00,1.154701e+01,1.000000e+01,4.000000e+00,5.500000e+00,9.155330e-14,NaN,NaN
7,C1,-1.2,0.30,1.5,1.035349e+01,1.000000e+04,1.638506e+02,7.115940e+01,3.214550e+02,1.000000e+03,3.100000e+01,4.600000e+01,9.382462e-13,2.107104e+00,NaN
8,C1,-1.2,0.30,1.5,1.100389e+01,1.000000e+06,1.521526e+03,6.607903e+02,1.001665e+04,1.000000e+05,3.010000e+02,4.510000e+02,1.221677e-13,6.503930e-01,0.308667
9,C1,-1.2,0.30,1.5,1.120907e+01,1.000000e+08,1.502920e+04,6.527099e+03,3.162805e+05,1.000000e+07,3.001000e+03,4.501000e+03,9.477450e-13,2.051855e-01,0.315479


In [59]:
growing_time_table = (
    growing_convergence[
        [
            "case_id",
            "rho_target",
            "t_target",
            "delta_t",
            "delta_ratio",
            "log10_a_target",
            "H_target",
            "Hdot_target",
        ]
    ]
    .copy()
)

with pd.option_context(
    "display.float_format",
    "{:.6e}".format,
):
    display(growing_time_table)

,case_id,rho_target,t_target,delta_t,delta_ratio,log10_a_target,H_target,Hdot_target
0,B3,1.000000e+02,5.924525e+00,NaN,NaN,4.333333e+00,6.582806e+00,1.000000e+01
1,B3,1.000000e+04,6.516978e+00,5.924525e-01,NaN,8.666667e+00,6.582806e+01,1.000000e+03
2,B3,1.000000e+06,6.576223e+00,5.924525e-02,1.000000e-01,1.300000e+01,6.582806e+02,1.000000e+05
3,B3,1.000000e+08,6.582148e+00,5.924525e-03,1.000000e-01,1.733333e+01,6.582806e+03,1.000000e+07
4,B3,1.000000e+10,6.582740e+00,5.924525e-04,1.000000e-01,2.166667e+01,6.582806e+04,1.000000e+09
5,B3,1.000000e+12,6.582799e+00,5.924525e-05,1.000000e-01,2.600000e+01,6.582806e+05,1.000000e+11
6,C1,1.000000e+02,8.246388e+00,NaN,NaN,9.196309e+00,1.154701e+01,1.000000e+01
7,C1,1.000000e+04,1.035349e+01,2.107104e+00,NaN,7.115940e+01,3.214550e+02,1.000000e+03
8,C1,1.000000e+06,1.100389e+01,6.503930e-01,3.086667e-01,6.607903e+02,1.001665e+04,1.000000e+05
9,C1,1.000000e+08,1.120907e+01,2.051855e-01,3.154792e-01,6.527099e+03,3.162805e+05,1.000000e+07


In [ ]:
expected_finite_time_cases = [
    "B3",
    "C1",
    "D2",
    "F1",
]

expected_logarithmic_time_case = [
    "C2",
]

expected_infinite_time_cases = [
    "C3",
]

In [62]:
growing_convergence_summary = (
    growing_convergence
    .groupby("case_id", as_index=False)
    .tail(1)
    [
        [
            "case_id",
            "rho_target",
            "t_target",
            "delta_t",
            "delta_ratio",
            "log10_a_target",
        ]
    ]
    .reset_index(drop=True)
)

growing_convergence_summary

,case_id,rho_target,t_target,delta_t,delta_ratio,log10_a_target
0,B3,1.000000e+12,6.582799e+00,5.924525e-05,0.10000,2.600000e+01
1,C1,1.000000e+12,1.129445e+01,2.051322e-02,0.31622,6.514611e+05
2,C2,1.000000e+12,8.806324e+01,1.456283e+01,1.00000,4.342945e+11
3,C3,1.000000e+12,9.486823e+06,8.538150e+06,10.00000,3.257209e+23
4,D2,1.000000e+12,5.791166e+00,5.196152e-05,0.10000,2.021715e+01
5,F1,1.000000e+12,5.371456e+00,5.196152e-05,0.10000,1.990952e+01


In [64]:
def preliminary_growth_behavior(row):
    ratio = row["delta_ratio"]

    if not np.isfinite(ratio):
        return "no concluyente"

    if ratio < 0.8:
        return "compatible con tiempo finito"

    if 0.8 <= ratio <= 1.2:
        return "compatible con crecimiento logarítmico del tiempo"

    return "compatible con tiempo infinito"

growing_convergence_summary[
    "preliminary_behavior"
] = growing_convergence_summary.apply(
    preliminary_growth_behavior,
    axis=1,
)

growing_convergence_summary

,case_id,rho_target,t_target,delta_t,delta_ratio,log10_a_target,preliminary_behavior
0,B3,1.000000e+12,6.582799e+00,5.924525e-05,0.10000,2.600000e+01,compatible con tiempo finito
1,C1,1.000000e+12,1.129445e+01,2.051322e-02,0.31622,6.514611e+05,compatible con tiempo finito
2,C2,1.000000e+12,8.806324e+01,1.456283e+01,1.00000,4.342945e+11,compatible con crecimiento logarítmico del tiempo
3,C3,1.000000e+12,9.486823e+06,8.538150e+06,10.00000,3.257209e+23,compatible con tiempo infinito
4,D2,1.000000e+12,5.791166e+00,5.196152e-05,0.10000,2.021715e+01,compatible con tiempo finito
5,F1,1.000000e+12,5.371456e+00,5.196152e-05,0.10000,1.990952e+01,compatible con tiempo finito


In [66]:
decreasing_density_targets_F2 = [
    1e-2,
    1e-4,
    1e-6,
    1e-8,
    1e-10,
    1e-12,
]

F2_convergence = scan_density_targets(
    case_id="F2",
    density_targets=decreasing_density_targets_F2,
)

with pd.option_context(
    "display.float_format",
    "{:.6e}".format,
):
    display(
        F2_convergence[
            [
                "case_id",
                "rho_target",
                "t_target",
                "delta_t",
                "delta_ratio",
                "log10_a_target",
                "H_target",
                "Hdot_target",
            ]
        ]
    )

,case_id,rho_target,t_target,delta_t,delta_ratio,log10_a_target,H_target,Hdot_target
0,F2,1.000000e-02,3.841370e+03,NaN,NaN,1.447170e+04,1.154715e+01,1.000000e-03
1,F2,1.000000e-04,3.848994e+06,3.845152e+06,NaN,1.447648e+08,1.154701e+02,1.000000e-05
2,F2,1.000000e-06,3.849002e+09,3.845153e+09,1.000000e+03,1.447648e+12,1.154701e+03,1.000000e-07
3,F2,1.000000e-08,3.849002e+12,3.845153e+12,1.000000e+03,1.447648e+16,1.154701e+04,1.000000e-09
4,F2,1.000000e-10,3.849002e+15,3.845153e+15,1.000000e+03,1.447648e+20,1.154701e+05,1.000000e-11
5,F2,1.000000e-12,3.849002e+18,3.845153e+18,1.000000e+03,1.447648e+24,1.154701e+06,1.000000e-13


In [67]:
finite_time_check = {
    case_id: (
        growing_convergence_summary
        .set_index("case_id")
        .loc[case_id, "delta_ratio"]
        < 0.8
    )
    for case_id in [
        "B3",
        "C1",
        "D2",
        "F1",
    ]
}

finite_time_check

{'B3': np.True_, 'C1': np.True_, 'D2': np.True_, 'F1': np.True_}

In [68]:
infinite_time_check = {
    "C2": (
        0.8
        <= growing_convergence_summary
        .set_index("case_id")
        .loc["C2", "delta_ratio"]
        <= 1.2
    ),
    "C3": (
        growing_convergence_summary
        .set_index("case_id")
        .loc["C3", "delta_ratio"]
        > 1.2
    ),
    "F2": (
        F2_convergence[
            "delta_t"
        ]
        .dropna()
        .is_monotonic_increasing
    ),
}

infinite_time_check

{'C2': np.True_, 'C3': np.True_, 'F2': True}

In [69]:
boundary_case_ids = [
    "C4",
    "D3",
    "E2",
    "F3",
    "F4",
]

boundary_tolerances = [
    1e-2,
    1e-3,
    1e-4,
    1e-5,
    1e-6,
    1e-7,
    1e-8,
    1e-9,
]

In [71]:
def compute_boundary_diagnostics(
    params,
    final_state,
):
    """
    Calcula cantidades auxiliares cerca de una frontera.

    Estas cantidades se utilizan únicamente para diagnóstico
    y no se incorporan como nuevas variables del background.
    """

    rho = float(final_state["rho"])
    a = float(final_state["a"])
    H = float(final_state["H"])
    Hdot = float(final_state["Hdot"])

    power = rho ** (
        params.beta - 1.0
    )

    F = (
        1.0
        + params.alpha * power
    )

    D = (
        1.0
        + params.alpha
        * params.beta
        * power
    )

    if np.isclose(D, 0.0, atol=1e-16):
        rho_dot = np.nan
    else:
        rho_dot = (
            -3.0
            * H
            * (1.0 + params.omega)
            * rho
            / D
        )

    Hddot = (
        -0.5
        * (1.0 + params.omega)
        * rho_dot
    )

    return {
        "rho_final": rho,
        "a_final": a,
        "H_final": H,
        "Hdot_final": Hdot,
        "F_final": float(F),
        "D_final": float(D),
        "rho_dot_final": float(rho_dot),
        "Hddot_final": float(Hddot),
    }

def scan_boundary_tolerances(
    case_id,
    tolerances,
):
    """
    Integra un caso repetidamente mientras boundary_tol
    se aproxima a cero.

    No presupone qué evento ni qué clasificación debe aparecer.
    """

    if case_id not in case_lookup:
        raise KeyError(
            f"No existe el caso {case_id}."
        )

    case = case_lookup[case_id]
    params = case["params"]

    rows = []

    for boundary_tol in tolerances:
        model = BarotropicPowerDiffusionModel(
            params
        )

        try:
            background = model.solve_background_t(
                t_end=100.0,
                n_points=2000,
                rho_min=1e-14,
                rho_max=1e50,
                a_max=1e100,
                boundary_tol=boundary_tol,
                method="DOP853",
                rtol=1e-10,
                atol=1e-12,
                max_step=0.01,
            )

            final_state = background.iloc[-1]

            diagnostics = (
                compute_boundary_diagnostics(
                    params=params,
                    final_state=final_state,
                )
            )

            critical = critical_density_summary(
                alpha=params.alpha,
                beta=params.beta,
            )

            rho_H = critical[
                "rho_H_boundary"
            ]

            rho_D = critical[
                "rho_denominator_boundary"
            ]

            rho_final = diagnostics[
                "rho_final"
            ]

            if np.isfinite(rho_H):
                relative_distance_H = (
                    abs(rho_final - rho_H)
                    / max(abs(rho_H), 1e-300)
                )
            else:
                relative_distance_H = np.nan

            if np.isfinite(rho_D):
                relative_distance_D = (
                    abs(rho_final - rho_D)
                    / max(abs(rho_D), 1e-300)
                )
            else:
                relative_distance_D = np.nan

            rows.append(
                {
                    "case_id": case_id,
                    "omega": params.omega,
                    "alpha": params.alpha,
                    "beta": params.beta,
                    "boundary_tol": boundary_tol,
                    "termination_event": (
                        background.attrs.get(
                            "termination_event"
                        )
                    ),
                    "solver_success": bool(
                        background.attrs.get(
                            "solver_success",
                            False,
                        )
                    ),
                    "t_final": float(
                        final_state["t"]
                    ),
                    "rho_H": rho_H,
                    "rho_D": rho_D,
                    "relative_distance_H": (
                        relative_distance_H
                    ),
                    "relative_distance_D": (
                        relative_distance_D
                    ),
                    **diagnostics,
                    "error": None,
                }
            )

        except Exception as exc:
            rows.append(
                {
                    "case_id": case_id,
                    "omega": params.omega,
                    "alpha": params.alpha,
                    "beta": params.beta,
                    "boundary_tol": boundary_tol,
                    "termination_event": None,
                    "solver_success": False,
                    "t_final": np.nan,
                    "rho_H": np.nan,
                    "rho_D": np.nan,
                    "relative_distance_H": np.nan,
                    "relative_distance_D": np.nan,
                    "rho_final": np.nan,
                    "a_final": np.nan,
                    "H_final": np.nan,
                    "Hdot_final": np.nan,
                    "F_final": np.nan,
                    "D_final": np.nan,
                    "rho_dot_final": np.nan,
                    "Hddot_final": np.nan,
                    "error": str(exc),
                }
            )

    return pd.DataFrame(rows)

In [72]:
boundary_tables = []

for case_id in boundary_case_ids:
    table = scan_boundary_tolerances(
        case_id=case_id,
        tolerances=boundary_tolerances,
    )

    boundary_tables.append(table)

boundary_convergence = pd.concat(
    boundary_tables,
    ignore_index=True,
)

In [73]:
boundary_convergence[
    boundary_convergence["error"].notna()
][
    [
        "case_id",
        "boundary_tol",
        "error",
    ]
]

,case_id,boundary_tol,error
6,C4,1.000000e-08,El integrador no logró completar la evolución:...
7,C4,1.000000e-09,El integrador no logró completar la evolución:...
30,F3,1.000000e-08,El integrador no logró completar la evolución:...
31,F3,1.000000e-09,El integrador no logró completar la evolución:...


In [74]:
boundary_result_table = (
    boundary_convergence[
        [
            "case_id",
            "boundary_tol",
            "termination_event",
            "solver_success",
            "t_final",
            "rho_final",
            "a_final",
            "H_final",
            "Hdot_final",
            "F_final",
            "D_final",
            "rho_dot_final",
            "Hddot_final",
            "relative_distance_H",
            "relative_distance_D",
        ]
    ]
    .copy()
)

with pd.option_context(
    "display.float_format",
    "{:.6e}".format,
):
    display(boundary_result_table)

,case_id,boundary_tol,termination_event,solver_success,t_final,rho_final,a_final,H_final,Hdot_final,F_final,D_final,rho_dot_final,Hddot_final,relative_distance_H,relative_distance_D
0,C4,1.000000e-02,rho_dot_denominator_boundary,True,1.419932e+00,4.840000e+00,2.289174e+00,7.406304e-01,4.840000e-01,3.400000e-01,1.000000e-02,2.150791e+02,2.150791e+01,5.644000e-01,1.990000e-02
1,C4,1.000000e-03,rho_dot_denominator_boundary,True,1.420156e+00,4.928400e+00,2.289555e+00,7.407396e-01,4.928400e-01,3.340000e-01,1.000000e-03,2.190397e+03,2.190397e+02,5.564440e-01,1.999000e-03
2,C4,1.000000e-04,rho_dot_denominator_boundary,True,1.420159e+00,4.937284e+00,2.289558e+00,7.407407e-01,4.937284e-01,3.334000e-01,1.000000e-04,2.194348e+04,2.194348e+03,5.556444e-01,1.999900e-04
3,C4,1.000000e-05,rho_dot_denominator_boundary,True,1.420159e+00,4.938173e+00,2.289558e+00,7.407407e-01,4.938173e-01,3.333400e-01,9.999999e-06,2.194744e+05,2.194744e+04,5.555644e-01,1.999990e-05
4,C4,1.000000e-06,rho_dot_denominator_boundary,True,1.420159e+00,4.938262e+00,2.289558e+00,7.407407e-01,4.938262e-01,3.333340e-01,1.000016e-06,2.194749e+06,2.194749e+05,5.555564e-01,2.000030e-06
5,C4,1.000000e-07,rho_dot_denominator_boundary,True,1.420159e+00,4.938271e+00,2.289558e+00,7.407407e-01,4.938271e-01,3.333334e-01,1.003732e-07,2.186625e+07,2.186625e+06,5.555556e-01,2.007465e-07
6,C4,1.000000e-08,None,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,C4,1.000000e-09,None,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,D3,1.000000e-02,effective_density_boundary,True,1.673449e+00,2.550760e-01,1.339189e+00,2.915910e-02,-1.275380e-01,1.000000e-02,5.050000e-01,-4.418488e-02,2.209244e-02,2.030405e-02,3.081216e+00
9,D3,1.000000e-03,effective_density_boundary,True,1.832105e+00,2.505008e-01,1.343258e+00,9.137847e-03,-1.252504e-01,1.000000e-03,5.005000e-01,-1.372050e-02,6.860252e-03,2.003004e-03,3.008012e+00


In [75]:
def relative_spread(values):
    """
    Dispersión relativa de una secuencia de valores finitos.
    """

    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if values.size < 2:
        return np.nan

    scale = max(
        abs(np.mean(values)),
        1e-300,
    )

    return float(
        (
            np.max(values)
            - np.min(values)
        )
        / scale
    )

In [76]:
boundary_summary_rows = []

for case_id, group in boundary_convergence.groupby(
    "case_id"
):
    group = (
        group
        .sort_values(
            "boundary_tol",
            ascending=False,
        )
        .dropna(
            subset=["t_final"]
        )
    )

    tail = group.tail(3)

    last = group.iloc[-1]

    previous = (
        group.iloc[-2]
        if len(group) >= 2
        else None
    )

    if previous is not None:
        rho_dot_growth = (
            abs(last["rho_dot_final"])
            / max(
                abs(previous["rho_dot_final"]),
                1e-300,
            )
        )

        Hddot_growth = (
            abs(last["Hddot_final"])
            / max(
                abs(previous["Hddot_final"]),
                1e-300,
            )
        )
    else:
        rho_dot_growth = np.nan
        Hddot_growth = np.nan

    boundary_summary_rows.append(
        {
            "case_id": case_id,
            "last_event": last[
                "termination_event"
            ],
            "smallest_boundary_tol": last[
                "boundary_tol"
            ],
            "t_limit_estimate": last[
                "t_final"
            ],
            "rho_limit_estimate": last[
                "rho_final"
            ],
            "a_limit_estimate": last[
                "a_final"
            ],
            "H_limit_estimate": last[
                "H_final"
            ],
            "Hdot_limit_estimate": last[
                "Hdot_final"
            ],
            "F_last": last["F_final"],
            "D_last": last["D_final"],
            "rho_dot_last": last[
                "rho_dot_final"
            ],
            "Hddot_last": last[
                "Hddot_final"
            ],
            "relative_time_spread": (
                relative_spread(
                    tail["t_final"]
                )
            ),
            "relative_rho_spread": (
                relative_spread(
                    tail["rho_final"]
                )
            ),
            "rho_dot_growth": (
                rho_dot_growth
            ),
            "Hddot_growth": (
                Hddot_growth
            ),
        }
    )

boundary_convergence_summary = pd.DataFrame(
    boundary_summary_rows
)

with pd.option_context(
    "display.float_format",
    "{:.6e}".format,
):
    display(boundary_convergence_summary)

,case_id,last_event,smallest_boundary_tol,t_limit_estimate,rho_limit_estimate,a_limit_estimate,H_limit_estimate,Hdot_limit_estimate,F_last,D_last,rho_dot_last,Hddot_last,relative_time_spread,relative_rho_spread,rho_dot_growth,Hddot_growth
0,C4,rho_dot_denominator_boundary,1.000000e-07,1.420159e+00,4.938271e+00,2.289558e+00,7.407407e-01,4.938271e-01,3.333334e-01,1.003732e-07,2.186625e+07,2.186625e+06,1.584931e-10,1.979930e-05,9.962989e+00,9.962989e+00
1,D3,effective_density_boundary,1.000000e-09,1.905086e+00,2.500000e-01,1.343706e+00,9.128709e-06,-1.250000e-01,1.000000e-09,5.000000e-01,-1.369306e-05,6.846532e-06,3.454437e-04,1.980000e-07,3.162278e-01,3.162278e-01
2,E2,effective_density_boundary,1.000000e-09,1.282498e+00,5.000000e-01,1.259921e+00,1.290995e-05,-2.500000e-01,1.000000e-09,1.000000e+00,-1.936492e-05,9.682459e-06,3.625276e-04,9.900001e-08,3.162278e-01,3.162278e-01
3,F3,rho_dot_denominator_boundary,1.000000e-07,1.743682e-01,5.000000e-01,1.111876e+00,5.773503e-01,-2.500000e-01,2.000000e+00,1.046678e-07,-8.274040e+06,4.137020e+06,8.276776e-11,4.947689e-06,9.554865e+00,9.554865e+00
4,F4,effective_density_boundary,1.000000e-09,1.032763e+00,7.071068e-01,1.220009e+00,1.535260e-05,-3.535534e-01,9.999999e-10,2.000000e+00,-1.628389e-05,8.141944e-06,3.740610e-04,4.950000e-08,3.162277e-01,3.162277e-01


In [77]:
provisional_behavior_labels = [
    "regular_expansion",
    "de_sitter",
    "finite_time_divergence",
    "infinite_time_divergence",
    "diffusion_driven_infinite_time_rip",
    "higher_derivative_boundary",
    "turnaround_candidate",
    "domain_boundary",
    "numerically_inconclusive",
]